# 🎙️ Whisper Tiny — Chinese Accent Fine-Tuning (ASCEND Dataset)

This notebook fine-tunes OpenAI's Whisper Tiny on Chinese-accented English speech
using the **ASCEND** dataset (A Spontaneous Chinese-English Dataset).

We use only the `language == 'en'` subset — 2,331 samples (~1.65 hours) of
Chinese native speakers speaking English. This is enough to run comfortably
on free Google Colab (T4 GPU).

### Goal:
Compare WER of **baseline Whisper Tiny** vs **fine-tuned Whisper Tiny** on
Chinese-accented English to see whether fine-tuning actually helps.

### Workflow:
1. Check GPU
2. Install dependencies
3. Load ASCEND English subset via streaming
4. Baseline evaluation — measure WER before fine-tuning
5. Build train/val datasets
6. Prepare features
7. Data collator and metrics
8. Configure experiments
9. Run fine-tuning
10. Compare results
11. Final WER comparison — fine-tuned vs baseline
12. Download best model

---
**Before running:** Runtime > Change runtime type > T4 GPU

## Step 0 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ No GPU found. Go to Runtime > Change runtime type > T4 GPU')

Mon Mar  2 05:07:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 1 — Install Dependencies

In [ ]:
!apt-get install -q ffmpeg
!pip install -q transformers datasets librosa soundfile jiwer accelerate evaluate scikit-learn
print('✅ Dependencies installed')

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 68.5 MB/s eta 0:00:00
✅ Dependencies installed


## Step 2 — Load ASCEND English Subset

ASCEND is a Chinese-English spontaneous speech dataset collected in Hong Kong.
We filter to `language == 'en'` only — these are Chinese native speakers speaking English.

- Total English samples: **2,331**
- Total English duration: **~1.65 hours**
- Streaming mode: loads samples on the fly, no full download needed ✅

In [ ]:
from datasets import load_dataset, Audio
import numpy as np

TARGET_SR = 16000

print('🔄 Loading ASCEND dataset (streaming)...')
ascend_stream = load_dataset('CAiRE/ASCEND', streaming=True, trust_remote_code=True)

# Filter to English-only samples and collect into a list
# We collect into memory since 1.65 hours is small enough
print('🔍 Filtering English samples...')
en_samples = []
total_duration = 0

for sample in ascend_stream['train']:
    if sample['language'] == 'en':
        en_samples.append({
            'array': sample['audio']['array'].astype(np.float32),
            'sampling_rate': sample['audio']['sampling_rate'],
            'sentence': sample['transcription']
        })
        total_duration += sample['duration']

print(f'\n✅ Collected {len(en_samples)} English samples')
print(f'⏱️  Total duration: {total_duration/3600:.2f} hours')
print(f'\n🔍 Sample entry:')
print(f'   Transcript : "{en_samples[0]["sentence"]}"')
print(f'   Array shape: {en_samples[0]["array"].shape}')
print(f'   Sample rate: {en_samples[0]["sampling_rate"]}')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'CAiRE/ASCEND' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'CAiRE/ASCEND' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


🔄 Loading ASCEND dataset (streaming)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

🔍 Filtering English samples...

✅ Collected 2331 English samples
⏱️  Total duration: 1.65 hours

🔍 Sample entry:
   Transcript : "we have the sea shore in the city and we have a lot of"
   Array shape: (65920,)
   Sample rate: 16000


## Step 3 — Resample to 16kHz

ASCEND audio may not all be at 16kHz. Whisper requires exactly 16kHz.
We resample anything that isn't already correct.

In [ ]:
import librosa

print('🔄 Resampling audio to 16kHz where needed...')
resampled = 0

for sample in en_samples:
    if sample['sampling_rate'] != TARGET_SR:
        sample['array'] = librosa.resample(
            sample['array'],
            orig_sr=sample['sampling_rate'],
            target_sr=TARGET_SR
        )
        sample['sampling_rate'] = TARGET_SR
        resampled += 1

print(f'✅ Done. {resampled} samples resampled, {len(en_samples) - resampled} already at 16kHz.')

🔄 Resampling audio to 16kHz where needed...
✅ Done. 0 samples resampled, 2331 already at 16kHz.


## Step 4 — Baseline Evaluation (Before Fine-Tuning)

Before training anything, we measure how well stock Whisper Tiny performs
on Chinese-accented English. This is our baseline WER to beat.

We evaluate on a random sample of 200 clips to keep it fast.

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import torch
import evaluate
import random

wer_metric = evaluate.load('wer')

print('🔄 Loading baseline Whisper Tiny...')
baseline_processor = WhisperProcessor.from_pretrained('openai/whisper-tiny.en')
baseline_model = WhisperForConditionalGeneration.from_pretrained('openai/whisper-tiny.en')
baseline_model.eval()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
baseline_model = baseline_model.to(device)
print(f'✅ Model loaded on {device}')

# Sample 200 random clips for baseline eval
BASELINE_EVAL_N = 200
eval_subset = random.sample(en_samples, BASELINE_EVAL_N)

print(f'\n🔄 Running baseline evaluation on {BASELINE_EVAL_N} samples...')
predictions = []
references = []

for i, sample in enumerate(eval_subset):
    inputs = baseline_processor(
        sample['array'],
        sampling_rate=TARGET_SR,
        return_tensors='pt'
    ).input_features.to(device)

    with torch.no_grad():
        generated_ids = baseline_model.generate(inputs)

    pred = baseline_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    predictions.append(pred)
    references.append(sample['sentence'])

    if (i + 1) % 50 == 0:
        print(f'  {i+1}/{BASELINE_EVAL_N} done...')

baseline_wer = 100 * wer_metric.compute(predictions=predictions, references=references)

print(f'\n📊 Baseline Whisper Tiny WER on Chinese-accented English: {baseline_wer:.2f}%')
print('\n🔍 Sample predictions:')
for i in range(3):
    print(f'  REF : "{references[i]}"')
    print(f'  PRED: "{predictions[i]}"')
    print()

🔄 Loading baseline Whisper Tiny...


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

generation_config.json: 0.00B [00:00, ?B/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


✅ Model loaded on cuda

🔄 Running baseline evaluation on 200 samples...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

  50/200 done...
  100/200 done...
  150/200 done...
  200/200 done...

📊 Baseline Whisper Tiny WER on Chinese-accented English: 49.07%

🔍 Sample predictions:
  REF : "because it help them to grow up mentally healthier"
  PRED: " because it helps them to grow up mentally healthier."

  REF : "so you have still having the interview"
  PRED: " still having the interview."

  REF : "it's kind of like one part of my"
  PRED: " It's kind of like when part of my"



## Step 5 — Train / Val Split

We split the 2,331 English samples into:
- **85% train** (~1,981 samples)
- **15% val** (~350 samples)

The val set is held out and never seen during training — used only to measure WER per epoch.

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

indices = list(range(len(en_samples)))
train_idx, val_idx = train_test_split(indices, test_size=0.15, random_state=42)

def make_dataset(idx_list):
    return Dataset.from_dict({
        'audio': [
            {'array': en_samples[i]['array'], 'sampling_rate': TARGET_SR}
            for i in idx_list
        ],
        'sentence': [en_samples[i]['sentence'] for i in idx_list]
    })

train_dataset = make_dataset(train_idx)
val_dataset   = make_dataset(val_idx)

print(f'✅ Train samples : {len(train_dataset)}')
print(f'✅ Val samples   : {len(val_dataset)}')

✅ Train samples : 1981
✅ Val samples   : 350


## Step 6 — Load Processor and Prepare Features

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained('openai/whisper-tiny.en')

def prepare_dataset(batch):
    audio = batch['audio']
    batch['input_features'] = processor(
        audio['array'],
        sampling_rate=audio['sampling_rate'],
        return_tensors='pt'
    ).input_features[0]
    batch['labels'] = processor.tokenizer(batch['sentence']).input_ids
    return batch

print('🔄 Preparing features...')
train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names)
val_dataset   = val_dataset.map(prepare_dataset, remove_columns=val_dataset.column_names)
print('✅ Features ready')

🔄 Preparing features...


Map:   0%|          | 0/1981 [00:00<?, ? examples/s]

Map:   0%|          | 0/350 [00:00<?, ? examples/s]

✅ Features ready


## Step 7 — Data Collator and Metrics

In [ ]:
import torch
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')
        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

metric = evaluate.load('wer')

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {'wer': wer}

print('✅ Collator and metrics ready')

✅ Collator and metrics ready


## Step 8 — Experiment Configuration

Three experiments with increasing aggressiveness.
Given 1.65 hours of data, we keep epochs conservative to avoid overfitting.

- **exp1_conservative** — safe starting point
- **exp2_moderate** — balanced
- **exp3_aggressive** — higher risk of overfitting on this dataset size

In [ ]:
EXPERIMENTS = [
    {
        'name': 'exp1_conservative',
        'learning_rate': 1e-5,
        'num_train_epochs': 3,
        'per_device_train_batch_size': 16,
        'warmup_steps': 10,
    },
    {
        'name': 'exp2_moderate',
        'learning_rate': 5e-5,
        'num_train_epochs': 5,
        'per_device_train_batch_size': 16,
        'warmup_steps': 20,
    },
    {
        'name': 'exp3_aggressive',
        'learning_rate': 1e-4,
        'num_train_epochs': 8,
        'per_device_train_batch_size': 16,
        'warmup_steps': 30,
    },
]

print(f'🧪 {len(EXPERIMENTS)} experiments configured:')
for exp in EXPERIMENTS:
    print(f"  → {exp['name']:<25} | LR: {exp['learning_rate']} | Epochs: {exp['num_train_epochs']}")

🧪 3 experiments configured:
  → exp1_conservative         | LR: 1e-05 | Epochs: 3
  → exp2_moderate             | LR: 5e-05 | Epochs: 5
  → exp3_aggressive           | LR: 0.0001 | Epochs: 8


## Step 9 — Run All Experiments

In [ ]:
from transformers import (
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import json

all_results = []

for exp in EXPERIMENTS:
    print('\n' + '='*60)
    print(f"🚀 Running: {exp['name']}")
    print(f"   LR: {exp['learning_rate']} | Epochs: {exp['num_train_epochs']}")
    print('='*60)

    model = WhisperForConditionalGeneration.from_pretrained('openai/whisper-tiny.en')
    model.generation_config.forced_decoder_ids = None
    model.generation_config.suppress_tokens = []

    output_dir = f"/content/whisper-chinese-{exp['name']}"

    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=exp['per_device_train_batch_size'],
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=1,
        learning_rate=exp['learning_rate'],
        warmup_steps=exp['warmup_steps'],
        num_train_epochs=exp['num_train_epochs'],
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='wer',
        greater_is_better=False,
        predict_with_generate=True,
        generation_max_length=225,
        logging_steps=5,
        report_to=['none'],
        fp16=True,
        dataloader_num_workers=2,
    )

    trainer = Seq2SeqTrainer(
        args=training_args,
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        processing_class=processor.feature_extractor,
    )

    trainer.train()
    trainer.save_model(output_dir)
    processor.save_pretrained(output_dir)

    eval_results = trainer.evaluate()
    wer = eval_results.get('eval_wer', None)

    result = {
        'experiment': exp['name'],
        'learning_rate': exp['learning_rate'],
        'epochs': exp['num_train_epochs'],
        'wer': round(wer, 2) if wer else 'N/A',
        'model_path': output_dir
    }
    all_results.append(result)
    print(f"\n✅ {exp['name']} done | WER: {result['wer']}%")

with open('/content/experiment_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print('\n🎉 All experiments complete!')


🚀 Running: exp1_conservative
   LR: 1e-05 | Epochs: 3


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Wer
1,0.484615,0.560723,27.407407
2,0.486635,0.535887,26.840959
3,0.271901,0.535355,26.492375


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ exp3_aggressive done | WER: 28.32%

🎉 All experiments complete!


## Step 10 — Compare Experiment Results

In [ ]:
import json
import pandas as pd

with open('/content/experiment_results.json') as f:
    results = json.load(f)

results_df = pd.DataFrame(results)[['experiment', 'learning_rate', 'epochs', 'wer', 'model_path']]
results_df['wer'] = pd.to_numeric(results_df['wer'], errors='coerce')
results_df = results_df.sort_values('wer')

print('\n📊 Experiment Results (sorted by WER — lower is better):')
print('=' * 60)
print(results_df[['experiment', 'learning_rate', 'epochs', 'wer']].to_string(index=False))
print('=' * 60)

best = results_df.iloc[0]
print(f"\n🏆 Best experiment: {best['experiment']} with WER: {best['wer']}%")


📊 Experiment Results (sorted by WER — lower is better):
       experiment  learning_rate  epochs   wer
    exp2_moderate        0.00005       5 26.45
exp1_conservative        0.00001       3 26.49
  exp3_aggressive        0.00010       8 28.32

🏆 Best experiment: exp2_moderate with WER: 26.45%


## Step 11 — Final WER Comparison: Fine-Tuned vs Baseline

This is the key step — we run **both** the baseline and the best fine-tuned model
on the **same val set** and compare WER directly.

This tells us definitively whether fine-tuning on Chinese-accented English helped.

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Rebuild val samples from our split indices for inference
val_samples = [en_samples[i] for i in val_idx]

def evaluate_model(model, processor, samples, label='Model'):
    model.eval()
    model = model.to(device)
    preds = []
    refs  = []
    for i, sample in enumerate(samples):
        inputs = processor(
            sample['array'],
            sampling_rate=TARGET_SR,
            return_tensors='pt'
        ).input_features.to(device)
        with torch.no_grad():
            generated_ids = model.generate(inputs)
        pred = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        preds.append(pred)
        refs.append(sample['sentence'])
        if (i + 1) % 100 == 0:
            print(f'  [{label}] {i+1}/{len(samples)} evaluated...')
    wer = 100 * wer_metric.compute(predictions=preds, references=refs)
    return wer, preds, refs

# --- Baseline ---
print('🔄 Evaluating baseline Whisper Tiny on val set...')
baseline_model = WhisperForConditionalGeneration.from_pretrained('openai/whisper-tiny.en')
baseline_processor = WhisperProcessor.from_pretrained('openai/whisper-tiny.en')
baseline_wer_val, baseline_preds, refs = evaluate_model(
    baseline_model, baseline_processor, val_samples, label='Baseline'
)
print(f'\n📊 Baseline WER on val set: {baseline_wer_val:.2f}%')

# --- Best fine-tuned model ---
best_model_path = results_df.iloc[0]['model_path']
print(f'\n🔄 Evaluating best fine-tuned model ({results_df.iloc[0]["experiment"]})...')
finetuned_model = WhisperForConditionalGeneration.from_pretrained(best_model_path)
finetuned_processor = WhisperProcessor.from_pretrained(best_model_path)
finetuned_wer_val, finetuned_preds, _ = evaluate_model(
    finetuned_model, finetuned_processor, val_samples, label='Fine-tuned'
)
print(f'\n📊 Fine-tuned WER on val set: {finetuned_wer_val:.2f}%')

# --- Summary ---
improvement = baseline_wer_val - finetuned_wer_val
direction = '✅ Improved' if improvement > 0 else '❌ Got worse' if improvement < 0 else '➡️  No change'

print('\n' + '='*60)
print('📋 FINAL COMPARISON (on val set)')
print('='*60)
print(f'  Baseline Whisper Tiny : {baseline_wer_val:.2f}% WER')
print(f'  Fine-tuned Model      : {finetuned_wer_val:.2f}% WER')
print(f'  Difference            : {improvement:+.2f}% WER')
print(f'  Verdict               : {direction}')
print('='*60)

# --- Side by side samples ---
print('\n🔍 Sample predictions (first 5 val clips):')
for i in range(5):
    print(f'\n  REF      : "{refs[i]}"')
    print(f'  BASELINE : "{baseline_preds[i]}"')
    print(f'  FINETUNED: "{finetuned_preds[i]}"')

🔄 Evaluating baseline Whisper Tiny on val set...


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

  [Baseline] 100/350 evaluated...
  [Baseline] 200/350 evaluated...
  [Baseline] 300/350 evaluated...

📊 Baseline WER on val set: 50.68%

🔄 Evaluating best fine-tuned model (exp2_moderate)...


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

  [Fine-tuned] 100/350 evaluated...
  [Fine-tuned] 200/350 evaluated...
  [Fine-tuned] 300/350 evaluated...

📊 Fine-tuned WER on val set: 26.45%

📋 FINAL COMPARISON (on val set)
  Baseline Whisper Tiny : 50.68% WER
  Fine-tuned Model      : 26.45% WER
  Difference            : +24.23% WER
  Verdict               : ✅ Improved

🔍 Sample predictions (first 5 val clips):

  REF      : "but maybe it's not very"
  BASELINE : " but maybe it's not very..."
  FINETUNED: "but maybe it's not very"

  REF      : "I do agree"
  BASELINE : " I do agree."
  FINETUNED: "i do agree"

  REF      : "the battery is dying down much faster than before"
  BASELINE : " the battery is dying down much faster than before."
  FINETUNED: " the battery starting down much faster than before"

  REF      : "and also force on weekdays"
  BASELINE : " and also for some weekdays."
  FINETUNED: "and also for some weekdays"

  REF      : "I think i am not capable to evaluate"
  BASELINE : " I think I'm not capable to eval

## Step 12 — Download Best Model

In [ ]:
import shutil
from google.colab import files

best_path = results_df.iloc[0]['model_path']
zip_path  = '/content/best_chinese_accent_model.zip'

shutil.make_archive('/content/best_chinese_accent_model', 'zip', best_path)
print('📦 Model zipped. Downloading...')
files.download(zip_path)

📦 Model zipped. Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>